# 🚀 LoRA Fine-Tuning: Practical Implementation
## Visual LLM Educational Series - Module 2

Welcome to hands-on LoRA fine-tuning! In this notebook, you'll fine-tune a real language model using LoRA on a custom dataset.

### 🎯 What You'll Learn:
- ✅ Load and prepare a pre-trained model (Llama-2-7B)
- ✅ Apply LoRA configuration to the model
- ✅ Prepare a dataset for fine-tuning
- ✅ Train the model with LoRA
- ✅ Save and load LoRA adapters
- ✅ Test the fine-tuned model

### ⏱️ Estimated Time: 45 minutes
### 🎯 Difficulty: Intermediate
### 💻 Requirements: Google Colab Pro (for larger models) or T4 GPU

---

**🔗 Part of the Visual LLM Educational Platform**  
Visit: [Visual LLM Platform](https://visual-llm-platform.herokuapp.com) for the complete curriculum!

In [ ]:
# Quick environment setup
!pip install -q transformers==4.36.0 peft==0.7.1 datasets==2.14.0 accelerate==0.24.0 bitsandbytes==0.41.0 trl==0.7.4

import torch
print(f"🔥 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📱 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 🤖 Loading the Base Model

We'll use **Microsoft's DialoGPT-medium** for this tutorial (smaller and faster than Llama for learning purposes).

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import pandas as pd

# Model configuration
model_name = "microsoft/DialoGPT-medium"  # 345M parameters - perfect for learning

print(f"📥 Loading model: {model_name}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Use half precision for memory efficiency
    device_map="auto"           # Automatically place on GPU
)

print(f"✅ Model loaded successfully!")
print(f"📊 Model parameters: {model.num_parameters() / 1e6:.1f}M")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (fp16)")

## ⚙️ Applying LoRA Configuration

Now let's apply LoRA to make the model parameter-efficient!

In [ ]:
# LoRA Configuration
lora_config = LoraConfig(
    r=16,                    # Rank - controls adaptation capacity
    lora_alpha=32,           # Scaling factor
    target_modules=[         # Apply LoRA to attention layers
        "c_attn",           # Combined attention projection (DialoGPT specific)
        "c_proj",           # Output projection
    ],
    lora_dropout=0.1,        # Dropout for regularization
    bias="none",             # Don't adapt bias
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

print("\n🎯 LoRA Applied Successfully!")
print(f"📊 Original parameters: {model.base_model.num_parameters() / 1e6:.1f}M")
print(f"⚡ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")
print(f"💾 Memory reduction: ~{(1 - sum(p.numel() for p in model.parameters() if p.requires_grad) / model.base_model.num_parameters()) * 100:.1f}%")

## 📊 Dataset Preparation

Let's create a simple conversational dataset for fine-tuning our chatbot!

In [ ]:
# Create a sample dataset for LoRA fine-tuning
# This is a simple AI assistant dataset focused on LLM topics

conversations = [
    {
        "input": "What is LoRA?",
        "output": "LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique that reduces trainable parameters by up to 99% while maintaining model performance. It works by decomposing weight updates into low-rank matrices."
    },
    {
        "input": "How does LoRA work?",
        "output": "LoRA works by freezing the original model weights and adding trainable low-rank matrices A and B. Instead of updating the full weight matrix W, it updates W + A×B, where A and B are much smaller matrices."
    },
    {
        "input": "What are the benefits of LoRA?",
        "output": "LoRA offers several benefits: 99% reduction in trainable parameters, massive memory savings, faster training, cost-effective fine-tuning on consumer GPUs, and easy adapter merging and deployment."
    },
    {
        "input": "What is QLoRA?",
        "output": "QLoRA (Quantized LoRA) combines 4-bit quantization with LoRA to enable fine-tuning of even larger models like 70B parameters on single consumer GPUs. It uses techniques like NF4 quantization and double quantization."
    },
    {
        "input": "How to choose LoRA rank?",
        "output": "LoRA rank (r) controls adaptation capacity. Start with r=16 for most tasks. Higher ranks (32-64) give better quality but more parameters. Lower ranks (4-8) are more efficient but may limit adaptation."
    },
    {
        "input": "What is PEFT?",
        "output": "PEFT (Parameter-Efficient Fine-Tuning) refers to methods that fine-tune large models using only a small subset of parameters. Examples include LoRA, AdaLoRA, Prefix Tuning, and P-Tuning v2."
    }
]

# Format conversations for training
def format_conversation(example):
    """Format conversation for causal language modeling"""
    text = f"Human: {example['input']}\nAssistant: {example['output']}<|endoftext|>"
    return {"text": text}

# Create dataset
dataset = Dataset.from_pandas(pd.DataFrame(conversations))
dataset = dataset.map(format_conversation)

print(f"📊 Dataset created with {len(dataset)} examples")
print("\n📝 Sample conversation:")
print(dataset[0]['text'])

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    """Tokenize text for training"""
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=512,  # Adjust based on your needs
        return_overflowing_tokens=False,
    )

# Tokenize dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
)

print(f"✅ Dataset tokenized successfully!")
print(f"📊 Tokenized examples: {len(tokenized_dataset)}")
print(f"🔤 Sample tokens: {len(tokenized_dataset[0]['input_ids'])} tokens")

## 🏋️ Training Setup

Now let's configure the training parameters and start fine-tuning!

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./lora-finetuned-model",
    overwrite_output_dir=True,
    num_train_epochs=3,              # Number of training epochs
    per_device_train_batch_size=2,   # Batch size per device
    gradient_accumulation_steps=2,   # Accumulate gradients
    warmup_steps=10,                 # Warmup steps
    learning_rate=5e-4,              # Learning rate (higher for LoRA)
    fp16=True,                       # Use mixed precision
    logging_steps=1,                 # Log every step
    save_strategy="epoch",           # Save every epoch
    evaluation_strategy="no",        # No evaluation for this demo
    remove_unused_columns=False,     # Keep all columns
    dataloader_pin_memory=False,     # Disable pin memory for stability
)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We're doing causal LM, not masked LM
)

print("⚙️ Training configuration:")
print(f"   📊 Epochs: {training_args.num_train_epochs}")
print(f"   📦 Batch size: {training_args.per_device_train_batch_size}")
print(f"   📈 Learning rate: {training_args.learning_rate}")
print(f"   🔥 Mixed precision: {training_args.fp16}")

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("🚀 Starting LoRA fine-tuning...")
print("⏱️ This will take a few minutes...")

# Start training
trainer.train()

print("\n🎉 Training completed successfully!")
print("💾 Model saved to ./lora-finetuned-model")

## 🧪 Testing the Fine-Tuned Model

Let's test our LoRA fine-tuned model!

In [ ]:
# Test the fine-tuned model
def generate_response(prompt, max_length=100):
    """Generate response using the fine-tuned model"""
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()

# Test questions
test_questions = [
    "Human: What is LoRA?\nAssistant:",
    "Human: How does QLoRA work?\nAssistant:",
    "Human: What are the benefits of parameter-efficient fine-tuning?\nAssistant:"
]

print("🧪 Testing the fine-tuned model:")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n📝 Test {i}:")
    print(f"Question: {question.split('Assistant:')[0].replace('Human: ', '')}")
    
    response = generate_response(question, max_length=150)
    print(f"Response: {response}")
    print("-" * 40)

## 💾 Saving and Loading LoRA Adapters

One of the best features of LoRA is that you can save just the adapter weights!

In [ ]:
# Save LoRA adapters (only the small adapter weights, not the full model)
adapter_path = "./lora-adapters"
model.save_pretrained(adapter_path)

print(f"💾 LoRA adapters saved to: {adapter_path}")

# Check file sizes
import os

def get_folder_size(folder_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for filename in filenames:
            filepath = os.path.join(dirpath, filename)
            total_size += os.path.getsize(filepath)
    return total_size

adapter_size = get_folder_size(adapter_path) / (1024 * 1024)  # MB
print(f"📊 Adapter size: {adapter_size:.1f} MB")
print(f"🎯 Compare to full model: ~700 MB (DialoGPT-medium)")
print(f"💡 Space savings: {(1 - adapter_size/700) * 100:.1f}%")

In [ ]:
# Demonstrate loading adapters (useful for deployment)
from peft import PeftModel

print("🔄 Loading LoRA adapters...")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load LoRA adapters
model_with_adapters = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

print("✅ LoRA adapters loaded successfully!")
print("🎯 Model ready for inference with fine-tuned capabilities")

# Quick test
test_prompt = "Human: What is LoRA?\nAssistant:"
inputs = tokenizer.encode(test_prompt, return_tensors="pt").to(model_with_adapters.device)

with torch.no_grad():
    outputs = model_with_adapters.generate(
        inputs,
        max_length=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n🧪 Quick test response:")
print(response[len(test_prompt):].strip())

## 🎉 Congratulations!

You've successfully completed practical LoRA fine-tuning! Here's what you accomplished:

### ✅ What You Learned:
1. **Loaded a pre-trained model** (DialoGPT-medium)
2. **Applied LoRA configuration** with optimal parameters
3. **Prepared a custom dataset** for fine-tuning
4. **Fine-tuned the model** using LoRA
5. **Tested the fine-tuned model** with new prompts
6. **Saved and loaded LoRA adapters** for deployment

### 📊 Key Results:
- **Parameter Reduction**: ~99% fewer trainable parameters
- **Memory Efficiency**: Significant memory savings
- **Adapter Size**: Only ~few MB vs hundreds of MB for full model
- **Training Time**: Minutes instead of hours

### 🚀 Next Steps:

**Continue Learning:**
- **Next Notebook**: [03_QLoRA_Advanced_Techniques.ipynb](link-to-next)
- **Visual LLM Platform**: [Complete Curriculum](https://your-visual-llm-app.herokuapp.com)

**Try These Experiments:**
1. **Different Ranks**: Try r=4, 8, 32, 64 and compare results
2. **More Target Modules**: Add more layers to target_modules
3. **Larger Dataset**: Use a bigger dataset for better results
4. **Different Models**: Try with Llama-2-7B or other models

### 💡 Key Takeaways:
- LoRA makes fine-tuning **accessible and efficient**
- Small adapters can achieve **significant improvements**
- **Easy deployment** with adapter loading
- **Cost-effective** training on consumer hardware

---

**🎓 Ready to explore advanced techniques like QLoRA?**  
Continue your journey with the Visual LLM platform!